# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/damiannogueira/flyrank-ml-intership/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Colab setup

This setup securely loads the Hugging Face access token from Colab Secrets so the notebook can access the gated FlyRank warehouse without exposing the token in the code.

In [1]:
# Load the Hugging Face token securely from Colab Secrets.
# The token is never printed or stored in the notebook.

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [2]:
# Install DuckDB and open a connection to the remote warehouse.
# The Hugging Face token is registered securely without printing it.

%pip -q install duckdb

import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("DuckDB connection ready.")

DuckDB connection ready.


In [3]:
# Define the March 2026 warehouse partition used throughout the notebook.

MARCH_REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

## 1. Unit of analysis + time window

### Unit of analysis

In the warehouse source table, one row represents the daily performance of one pseudonymized content page for one pseudonymized client on one report date.

For the Refresh / Content Opportunity Scoring lane, these daily observations will later be aggregated at the content-page level so that each page can receive a review-priority score.

### Time window

For this assignment, I will work with the mid-panel month of March 2026 (`2026-03-01` to `2026-03-31`) from `fact_content_daily_performance`.

I am deliberately using a mid-panel month rather than the June 2026 sample because the final month should remain separate from development of future-outcome logic.

## 2. Fields: feature / label / context / excluded

### Features

The model will use up to five signals that are available before the review decision, based on March 2026 search and content performance data. These features will be defined and built in Section 3.

### Label / proxy

The provisional target is a future observed decline in page visibility or performance, measured in a later time window. This target is not available yet in the March feature window and must remain separate from the features.

### Context

`content_hash_id`, `client_hash_id`, and `report_date` are context fields. They are useful for grouping, joining, checking the data grain, and defining time windows, but they should not be learned by the model as predictive features.

### Excluded

I deliberately exclude label-derived or future-information fields from the feature set. In particular, fields used to define the future outcome must not be used as predictors, because that would create data leakage.

I also exclude the final June 2026 sample from development because it should remain a sealed future evaluation window.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
# Query 1: verify the daily table grain.
# One row should represent one client, one content item, and one report date.

grain_check = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        COUNT(*) AS row_count
    FROM {MARCH_REL}
    GROUP BY
        client_hash_id,
        content_hash_id,
        report_date
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()

grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,row_count


In [5]:
# Query 2: verify the row count and date span for March 2026.

count_window_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {MARCH_REL}
""").df()

count_window_check

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [6]:
# Query 3: verify GSC data availability.
# Keep only rows where search-performance data is explicitly available.

availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS rows_with_gsc_available
    FROM {MARCH_REL}
    WHERE gsc_data_available IS TRUE
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_with_gsc_available
0,3611061


### Five-feature frame

I will use information available from March 1 through March 21, 2026, before the review decision moment.

The five features are:

1. `impressions_21d` — knowable at the decision moment because it uses only GSC impressions observed before March 22.
2. `clicks_21d` — knowable at the decision moment because it uses only GSC clicks observed before March 22.
3. `ctr_21d` — knowable at the decision moment because it is calculated only from pre-decision clicks and impressions.
4. `avg_position_21d` — knowable at the decision moment because it summarizes search position observed before March 22.
5. `active_days_21d` — knowable at the decision moment because it counts only pre-decision days on which the page received impressions.

In [7]:
# Build exactly five pre-decision features from March 1-21.
# Only rows with explicitly available GSC data are used.

feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions_21d,
        SUM(gsc_clicks) AS clicks_21d,

        100.0 * SUM(gsc_clicks)
            / NULLIF(SUM(gsc_impressions), 0) AS ctr_21d,

        SUM(gsc_sum_position)
            / NULLIF(SUM(gsc_impressions), 0) AS avg_position_21d,

        COUNT(DISTINCT CASE
            WHEN gsc_impressions > 0 THEN report_date
        END) AS active_days_21d

    FROM {MARCH_REL}

    WHERE
        gsc_data_available IS TRUE
        AND report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING SUM(gsc_impressions) > 0
""").df()

feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions_21d,clicks_21d,ctr_21d,avg_position_21d,active_days_21d
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,63.0,0.0,0.000000,3.968254,18
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,6623.0,9.0,0.135890,8.389552,21
2,client_62f4a7e64f5e0096,content_e689bc511192751a,41.0,0.0,0.000000,5.048780,18
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,536.0,1.0,0.186567,5.121269,21
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,44.0,0.0,0.000000,15.750000,17


### Future outcome proxy

For this exercise, I will use March 22–31 as a later outcome window.

A page is provisionally labeled as declining when its observed impression rate in the later window is more than 20% lower than its impression rate during the March 1–21 feature window.

This is a directional proxy for the exercise, not a claim that a content refresh would cause recovery.

In [8]:
# Build the later outcome window from March 22-31.
# These values are observed after the feature window and must not be used as honest features.

future_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS future_impressions_10d

    FROM {MARCH_REL}

    WHERE
        gsc_data_available IS TRUE
        AND report_date BETWEEN DATE '2026-03-22' AND DATE '2026-03-31'

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

model_frame = feature_frame.merge(
    future_frame,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Compare the future daily impression rate with the pre-decision daily rate.
model_frame["future_change_pct"] = (
    (
        model_frame["future_impressions_10d"] / 10
        - model_frame["impressions_21d"] / 21
    )
    / (model_frame["impressions_21d"] / 21)
    * 100
)

# Provisional future decline proxy: more than a 20% drop.
model_frame["future_decline_target"] = (
    model_frame["future_change_pct"] < -20
).astype(int)

print(f"Pages with a future outcome: {len(model_frame):,}")
print(
    f"Future decline rate: "
    f"{model_frame['future_decline_target'].mean() * 100:.1f}%"
)

model_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages with a future outcome: 145,091
Future decline rate: 29.9%


,client_hash_id,content_hash_id,impressions_21d,clicks_21d,ctr_21d,avg_position_21d,active_days_21d,future_impressions_10d,future_change_pct,future_decline_target
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,63.0,0.0,0.000000,3.968254,18,14.0,-53.333333,1
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,6623.0,9.0,0.135890,8.389552,21,4226.0,33.996678,0
2,client_62f4a7e64f5e0096,content_e689bc511192751a,41.0,0.0,0.000000,5.048780,18,20.0,2.439024,0
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,536.0,1.0,0.186567,5.121269,21,169.0,-33.787313,1
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,44.0,0.0,0.000000,15.750000,17,6.0,-71.363636,1


### Deliberate leakage experiment

First, I will train a quick model using only the five features that were available before the decision moment.

Then I will deliberately add `future_change_pct`, which is derived from the same future window used to define `future_decline_target`. This is leakage because it contains the answer in numerical form.

The leaked score is shown only to demonstrate the problem. `future_change_pct` will not be kept as a model feature.

In [9]:
# Compare an honest model with a deliberately leaked version.
# The train/test split is grouped by client so the same client
# does not appear in both sets.

from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier

honest_features = [
    "impressions_21d",
    "clicks_21d",
    "ctr_21d",
    "avg_position_21d",
    "active_days_21d"
]

target = "future_decline_target"

# Sort rows deterministically before creating the train/test split.
# This also gives tied model scores a reproducible row order.

model_frame = model_frame.sort_values(
    ["client_hash_id", "content_hash_id"]
).reset_index(drop=True)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        model_frame,
        model_frame[target],
        groups=model_frame["client_hash_id"]
    )
)

train_df = model_frame.iloc[train_idx]
test_df = model_frame.iloc[test_idx]

def precision_at_k(scores, labels, k=50):
    # Stable sorting makes tied prediction scores reproducible.
    top_k = (-scores).argsort(kind="stable")[:k]
    return labels.iloc[top_k].mean()

# Honest model: only pre-decision features.
honest_model = DecisionTreeClassifier(
    max_depth=3,
    class_weight="balanced",
    random_state=42
)

honest_model.fit(
    train_df[honest_features],
    train_df[target]
)

honest_scores = honest_model.predict_proba(
    test_df[honest_features]
)[:, 1]

honest_p50 = precision_at_k(
    honest_scores,
    test_df[target].reset_index(drop=True),
    50
)

# Deliberate leakage: add one future-derived feature.
leaky_features = honest_features + ["future_change_pct"]

leaky_model = DecisionTreeClassifier(
    max_depth=3,
    class_weight="balanced",
    random_state=42
)

leaky_model.fit(
    train_df[leaky_features],
    train_df[target]
)

leaky_scores = leaky_model.predict_proba(
    test_df[leaky_features]
)[:, 1]

leaky_p50 = precision_at_k(
    leaky_scores,
    test_df[target].reset_index(drop=True),
    50
)

print(f"Honest Precision@50: {honest_p50:.3f}")
print(f"Leaky Precision@50:  {leaky_p50:.3f}")

# Remove the leaked column from the feature list.
final_features = honest_features.copy()

print("Final feature count:", len(final_features))
print("Leakage column kept:", "future_change_pct" in final_features)

Honest Precision@50: 0.680
Leaky Precision@50:  1.000
Final feature count: 5
Leakage column kept: False


### Leakage result

The honest model achieved a Precision@50 of 0.680 using only the five pre-decision features.

After deliberately adding `future_change_pct`, which is derived from the future outcome window, Precision@50 increased to 1.000. This near-perfect result is not a genuine improvement; it is caused by data leakage because the leaked feature directly contains the information used to define the target.

The leaked column was removed, and the final feature set keeps only the five features available before the decision moment.

## 4. Data limits

One important limitation is that the warehouse is an unbalanced panel: different clients have different amounts of historical data available.

This means that a single fixed calendar window may not represent every client equally well. Some clients may have long search and analytics histories, while others may have much shorter coverage.

For this reason, results from the March 2026 slice should be interpreted as decision-support for the observed data, not as evidence that every client has the same amount or quality of historical information.

## Self-check

- [x] I defined the unit of analysis and the March 2026 time window.
- [x] I classified the fields as features, label/proxy, context, and excluded.
- [x] I verified the data grain with a real query.
- [x] I verified the March row count and date span.
- [x] I checked GSC availability using `IS TRUE`.
- [x] I built exactly five pre-decision features.
- [x] I explained when each feature is available.
- [x] I created a later outcome proxy without mixing future information into the honest features.
- [x] I demonstrated deliberate leakage and removed the leaked column afterward.
- [x] I named a limitation of the data slice.
- [x] My claims use careful, decision-support language.